In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from typing import TypedDict

class State(TypedDict):
    text: str
    category: str
    response: str

def classify(state):
    t = state["text"].lower()
    category = "technical" if any(x in t for x in ["code", "python", "bug"]) else "general"
    return {"category": category}

def route(state):
    return {}

def respond(state):
    return {"response": f"Handled as {state['category']}: {state['text']}"}

def human_review(state):
    decision = interrupt("Human review required. Approve or modify?")
    return {"response": decision}

g = StateGraph(State)
g.add_node("classify", classify)
g.add_node("route", route)
g.add_node("respond", respond)
g.add_node("human_review", human_review)

g.add_edge(START, "classify")
g.add_conditional_edges("classify", lambda s: s["category"],
                        {"technical": "route", "general": "respond"})
g.add_conditional_edges("route", lambda s: "human_review")
g.add_edge("human_review", END)
g.add_edge("respond", END)

graph = g.compile()

tests = [
    "Fix my Python bug", "How are you?", "Write code for sorting",
    "Tell me a joke", "Why is my code failing?"
]
for text in tests:
    print(graph.invoke({"text": text}))

# Resume after human input:
# graph.invoke(Command(resume="Approved"), config=config)


{'text': 'Fix my Python bug', 'category': 'technical', '__interrupt__': [Interrupt(value='Human review required. Approve or modify?', id='1f5774969bc10f314ace3660a112722c')]}
{'text': 'How are you?', 'category': 'general', 'response': 'Handled as general: How are you?'}
{'text': 'Write code for sorting', 'category': 'technical', '__interrupt__': [Interrupt(value='Human review required. Approve or modify?', id='f25aa60247cc42adb1071dc8adb4d21c')]}
{'text': 'Tell me a joke', 'category': 'general', 'response': 'Handled as general: Tell me a joke'}
{'text': 'Why is my code failing?', 'category': 'technical', '__interrupt__': [Interrupt(value='Human review required. Approve or modify?', id='db0579eddab1857052ee6e0f9cad5be0')]}
